# Learning Environmental Dynamics: Building Internal Models of a Simulated External Force

### **Abstract**

Motor adaptation is typically studied using limb perturbations in highly constrained tasks. However, everyday actions require compensating for uncertain environmental dynamics impacting manipulated objects within redundant execution spaces, where multiple combinations of motor variables can achieve success. To investigate this, we designed a novel virtual reality task where participants launched a ball across a lateral water current to spatially distinct targets. Two experiments (2.0 m/s and 3.0 m/s) tested if classic adaptation signatures remain consistent across varying perturbation sizes.  

Results revealed three key signatures. First, participants exhibited robust error reduction to all targets during training. Second, between-subjects analyses revealed that this learning produced substantial, immediate transfer to untrained target locations. Finally, persistent motor aftereffects appeared upon perturbation removal, despite explicit cues indicating absence of the perturbation. Together, these results demonstrate that humans form internal models under altered projectile–environment dynamics, and that prior experience facilitates broad generalization across the workspace. 

# Load Data & Formatting

In [1]:
# import key libraries
library(emmeans)
library(dplyr)
library(afex)
library(effsize)
# ("new")

# full dataset

# set working directory & load FULL data from local storage
setwd("C:/Users/jacob/water_current_MA/data/dual_generalization")
df_full <- read.csv("PCA_error_projectile_experiment _S1-projectile_experiment_S2-projectile_experiment_S1_3.0M-projectile_experiment 1_S2_3.0M-projectile_experiment 1_speed_neg3_0_neg2_0_0_0.csv")


# Format data types 
df_full$ppid_full <- factor(df_full$ppid_full)
df_full$speed_label <- factor(df_full$speed_label)
df_full$target_x_label <- factor(df_full$target_x_label)
df_full$phase <- factor(df_full$phase)

df_full$set_order <- factor(df_full$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_full$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_full$target_x_label <- factor(df_full$target_x_label, 
                                 levels = c("L60", "L30", "R30", "R60"))


# load the smaller, early late (first and last 2 trials/phase) dataset
df_early_late_2 <- read.csv("early_late_2_phase_PCA_error_projectile_experiment _S1-projectile_experiment_etc_speed_neg3_0_neg2_0.csv")

# Format data types 
df_early_late_2$ppid_full <- factor(df_early_late_2$ppid_full)
df_early_late_2$speed_label <- factor(df_early_late_2$speed_label)
df_early_late_2$target_x_label <- factor(df_early_late_2$target_x_label)
df_early_late_2$phase <- factor(df_early_late_2$phase)

df_early_late_2$set_order <- factor(df_early_late_2$set_order)
# change set_order names to reflect thesis, group 1 and group 2 labels
levels(df_early_late_2$set_order) <- c("group_1","group_2")

# ensure consistent levels
df_early_late_2$target_x_label <- factor(df_early_late_2$target_x_label, 
                                     levels = c("L60", "L30", "R30", "R60"))

Warning message:
"package 'emmeans' was built under R version 4.4.3"
Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'afex' was built under R version 4.4.3"
Loading required package: lme4

Warning message:
"package 'lme4' was built under R version 4.4.3"
Loading required package: Matrix

************
Welcome to afex. For support visit: http://afex.singmann.science/

- Functions for ANOVAs: aov_car(), aov_ez(), and aov_4()
- Methods for calculating p-values with mixed(): 'S', 'KR', 'LRT', and 'PB'
- 'afex_aov' and 'mixed' objects can be passed to emmeans() for follow-up tests
- Get and set global package options with: afex_options()
- Set sum-to-zero contrasts globally: set_sum_contrasts()
- For exa

# Training Phase 1: early and late 2 trials

Here we compared the mean minimum metric error for the first and last two trials per target during the naive, initial Training Phase 1. We conducted within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors. Each ANOVA was conducted across two target pair groups (i.e., Group 1 and Group 2) and separately for two water current experiments. Consequently, we ran a total of 4 within-subjects ANOVAs (two groups, two experiments). Holm's correction was reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine robust learning following practice compensating for the water current perturbation.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both targets within each pair produced comparable learning magnitudes from early to late training. However, the main effect of trial set was highly significant for both Group 1 and Group 2 after applying Holm's correction, producing large generalized eta-squared values of 0.245 and 0.311, respectively. This demonstrates a robust error reduction following practice compensating for the water current that was consistent across both target pair groups.


Speed 2.0 m/s

Crucially, the interaction in this experiment was significant in Group 1, but both targets showed significant error reduction--R60 drove larger errors as expected. Nonetheless, here we also observed significant main effects of trial set for both groups after applying Holm's correction. Effect sizes were similar to the previous experiment, with Group 1 yielding a large generalized eta-squared value of 0.355, while Group 2 showed a generalized eta-squared value of 0.311. Again, we observed a significant learning effect following practice in compensating for the water current.

ANOVA results are below.

In [3]:
# Isolate training phase
df_early_late_2_t1 <- df_early_late_2[df_early_late_2$phase == 'training_1',]
speeds_list <- unique(df_early_late_2_t1$speed_label)
set_order_list <- unique(df_early_late_2_t1$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_early_late_2_t1[df_early_late_2_t1$speed_label == speed & df_early_late_2_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.2255671 
Holm-Adjusted trial_set Main Effect p-value:  1.964611e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 333.79 49.81 *** .260   <.001
2                trial_set 1, 19 411.89 50.10 *** .304   <.001
3 target_x_label:trial_set 1, 19 288.23      1.57 .009    .226
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     32.1 4.54 19   7.078  <.0001

Results are averaged over the levels of: target_x_label 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.1702784 
Holm-Adjusted trial_set Main Effect p-value:  4.587144e-06 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE        

# Training Phase 2: early and late 2 trials

Similar to the analysis above, we conducted a within-subjects ANOVA on mean minimum metric error for the first and last two trials per target, but now during the second training phase. Again, we conducted four within-subjects ANOVAs on minimum metric error with trial set and target as within-subject factors, separately for both water current experiments. Holm's correction was also reported for the two group ANOVAs in each separate experiment. The purpose of this analysis was to examine any ongoing learning after completing Training Phase 1.


Speed 3.0 m/s

The interaction was not significant across either group, suggesting that both novel targets within each pair produced comparable learning magnitudes across this phase. Importantly, because the main effect of trial set was not significant, this demonstrates that learning had saturated by the onset of this phase. This hints at a transfer effect, where learning in the first training phase results in error saturation for novel targets in the second phase. Overall, because we did not observe a main effect or interaction involving trial set during Training Phase 2 for this experiment, this illustrates learning saturation, and thus no ongoing learning was detected.


Speed 2.0 m/s

Interestingly, the interaction in this experiment was significant for both groups. Post-hoc analyses revealed that this effect was driven by the more challenging, upstream rightward targets. However, only the R60 target in Group 2 remained significant following the post-hocs. This demonstrates a continuation of learning for arguably the most challenging target in this task, in this group. Because participants who experienced this target during the second training phase must have switched from R30 in Training Phase 1. This apparent ongoing learning could be driven by transitioning from an easier to a more challenging target, as the other group only showed a small trend with the R30 target after post-hocs. Overall, because the majority of main effects or interactions involving trial set were not significant—besides R60 in Group 2—this demonstrates that, like the other experiment, learning had mostly saturated by the onset of this phase. However, it is important to acknowledge that R30 approached significance in the Group 1 comparisson. This may suggest that throwing against the current at 2.0 m/s was ultimately a greater challenge relative to throwing with the current at this perturbation magnitude; thereby driving much larger initial errors for upstream targets compared to downstream targets. Perhaps raising the current to 3.0 m/s reasonably increased initial error thresholds more broadly across the workspace, resulting in broad learning saturation. 

ANOVA results are below.

In [9]:
# Isolate training phase
df_early_late_2_t2 <- df_early_late_2[df_early_late_2$phase == 'training_2',]
speeds_list <- unique(df_early_late_2_t2$speed_label)
set_order_list <- unique(df_early_late_2_t2$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_experiment_id_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_early_late_2_t2[df_early_late_2_t2$speed_label == speed & df_early_late_2_t2$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }


ANALYSIS FOR SPEED x GROUP: -3 group_2 
Holm-Adjusted Interaction p-value:  0.3581706 
Holm-Adjusted trial_set Main Effect p-value:  0.0001596359 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE         F  ges p.value
1           target_x_label 1, 19 375.12 34.55 *** .235   <.001
2                trial_set 1, 19 544.53      1.40 .018    .251
3 target_x_label:trial_set 1, 19 242.07      1.95 .011    .179
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '+' 0.1 ' ' 1

Non-Sig Interaction. Main Effect of trial_set:
 contrast     estimate   SE df t.ratio p.value
 early - late     6.18 5.22 19   1.184  0.2509

Results are averaged over the levels of: target_x_label 

ANALYSIS FOR SPEED x GROUP: -3 group_1 
Holm-Adjusted Interaction p-value:  0.3581706 
Holm-Adjusted trial_set Main Effect p-value:  1.296115e-05 
Anova Table (Type 3 tests)

Response: flip_min_distance_xPCA_mean_bc
                    Effect    df    MSE        

# Between-Subjects Transfer Test

Here, we were interested in comparing initial errors for each target during Training Phase 1 versus Training Phase 2. As a reminder, participants were exposed to a counterbalanced target pair during each phase. This design allows us to compare initial target errors when participants are naive to the perturbation (i.e., early trials of Training Phase 1) to those when they have already experienced the impact of the perturbation (i.e., early Training Phase 2), albeit in a spatially distinct region of the workspace. The purpose of this analysis was to measure generalization and transfer. Transfer would be observed if initial target errors during Training Phase 2 are significantly lower than initial errors during Training Phase 1. This would suggest participants have developed a generalizable internal model of the ball-water interactions, allowing them to generate motor commands closer to task success than their naive counterparts. To address this hypothesis, we decided to conduct a series of four one-tailed Welch's t-tests for both experiments, since our main question involved a between-subjects effect of phase across each target. We chose a one-tailed test because we have theoretical justification to expect a directional learning effect. This approach not only aligns with our expectations, but also provides enhanced power for detecting transfer. Again, Holm's correction was applied due to multiple comparisons.

Because experienced participants outperformed naive participants in all four t-tests for both water current experiments, this undeniably demonstrates strong transfer. Importantly, near-complete transfer (i.e., errors near 0) was observed for most of the downstream targets in both experiments, while strong but incomplete transfer was observed for the upstream targets. Nonetheless, the consistent and strong transfer effect observed broadly across the workspace provides evidence for a generalizable internal representation of ball-water dynamics, facilitating immediate error reduction to novel target locations.

One-tailed Welch's t-test results are below. 

In [11]:

# Filter for Transfer (Early trials only)
df_transfer <- df_early_late_2[df_early_late_2$trial_set == 'early',]
df_transfer$phase <- factor(df_transfer$phase, levels = c("training_1", "training_2"))

speeds_list <- unique(df_transfer$speed_label)
target_list <- unique(df_transfer$target_x_label)



for (s in speeds_list) {

    p_vals <- c()
    
    for (t in target_list) {
  
          cat("\n--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET:", s, t, "---\n")
          
          df_subset <- df_transfer[df_transfer$speed_label == s & df_transfer$target_x_label == t,]
          df_subset <- droplevels(df_subset) 
          
          res <- t.test(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset, alternative = "greater")
          print(res)

          # show sd
          print(tapply(df_subset$flip_min_distance_xPCA_mean_bc, df_subset$phase, sd, na.rm = TRUE))
        
          # effect sizes
          print(cohen.d(flip_min_distance_xPCA_mean_bc ~ phase, data = df_subset))
        
          # Grab p-value & store
          p_vals <- c(p_vals, res$p.value)

    }
    
    # show p-vals
    cat(' Raw p-vals for',s,'experiment:\n')
    print(p_vals)
    
    # adjusted p-vals
    cat(' Adjusted p-vals for',s,'experiment:\n')
    adj_p <- p.adjust(p_vals, method = "holm")
    print(adj_p)
}



--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L60 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 5.2619, df = 70.811, p-value = 7.264e-07
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 23.32759      Inf
sample estimates:
mean in group training_1 mean in group training_2 
               31.048690                -3.093021 

training_1 training_2 
  34.03668   25.63034 

Cohen's d

d estimate: 1.151436 (large)
95 percent confidence interval:
    lower     upper 
0.6966743 1.6061984 


--- RUNNING BETWEEN TRANSFER ANALYSIS FOR SPEED x TARGET: -3 L30 ---

	Welch Two Sample t-test

data:  flip_min_distance_xPCA_mean_bc by phase
t = 2.4971, df = 82.454, p-value = 0.007256
alternative hypothesis: true difference in means between group training_1 and group training_2 is greater than 0
95 percent confidence interval:
 6.026549      Inf
sample estim